# Ionet Ray Service Basics

Ray services are generally used for large-scale GPU cluster programming, such as using dozens or hundreds of cards. Generally, you need to use distributed frameworks like Ray to implement this.

In contrast, ionet's virtual machines and containers are generally scenarios with 1 or a few graphics cards.

Of course, Ray programming requires some understanding of the Ray programming framework. Here, based on ionet's Ray service, we'll provide a brief introduction.

- Ionet's Ray service address: https://cloud.io.net/cloud/clusters
- Documentation address: https://docs.io.net/docs/deploy-ray-cluster

## Ray Framework Introduction

### One-Sentence Summary

**Ray is an open-source, general-purpose distributed computing framework** whose core goal is to enable developers to **extremely easily scale single-machine Python programs to run on large-scale computing clusters** without worrying about complex underlying system issues.

You can think of it as a **"distributed Python operating system"**.

-----

### What Pain Points Does Ray Solve?

1. **Python's Parallel Limitations**: Due to the existence of the Global Interpreter Lock (GIL), Python has difficulty truly utilizing multi-core CPUs for compute-intensive parallel processing on a single machine.
2. **Complexity of Distributed Programming**: When one machine's computing resources are insufficient and need to scale to multiple machines (clusters), developers face a series of extremely complex problems such as network communication, task scheduling, node failures, and state management. This usually requires specialized distributed systems engineers to handle.

**Ray's emergence is to let you harness the powerful computing power of entire computing clusters as easily as writing ordinary Python code.**

-----

### Ray's Two Core Concepts

Ray's magic mainly comes from the two simple yet powerful basic modules (APIs) it provides:

1. **Tasks (Remote Tasks)**:

   * **What it does**: Turns an ordinary Python **function** into a task that can be **asynchronously executed** on any node in the cluster.
   * **How to implement**: Simply add a decorator `@ray.remote` to your function.
   * **How to call**: When calling, change `my_function()` to `my_function.remote()`.
   * **Features**: This call immediately returns a "future object" (Object ID), the program continues to run, while Ray schedules and executes this function on some node in the background. You can later get the real result through `ray.get()`. This makes large-scale parallel computing extremely simple.

2. **Actors (Remote Actors)**:

   * **What it does**: Turns an ordinary Python **class** into a stateful service that can survive in the cluster.
   * **How to implement**: Similarly, add a decorator `@ray.remote` to your class.
   * **How to call**: When creating instances, change `MyClass()` to `MyClass.remote()`.
   * **Features**: Actors are very suitable for scenarios that need to maintain their own state, such as an environment simulator, a parameter server, or an inference service that loads a large model and needs to be called repeatedly.

-----

### Ray's Main Uses and Ecosystem

Ray is not just a core framework, it also has a powerful native library ecosystem specifically designed to solve various large-scale problems in machine learning and AI:

* **Ray Tune**: Distributed **hyperparameter tuning**. Can simultaneously launch hundreds or thousands of training tasks with different parameters and manage them efficiently.
* **Ray Train**: Distributed **model training**. Can easily scale single-machine training code for PyTorch, TensorFlow, etc. to parallel training on multiple machines and multiple GPUs.
* **Ray Serve**: Distributed **model serving**. Can deploy trained models as highly available, scalable online inference services (APIs).
* **Ray Data**: Distributed **data processing**. Provides data loading and preprocessing capabilities for large-scale machine learning tasks.

These libraries together constitute an end-to-end, distributed solution that covers the entire lifecycle of AI applications.

### Summary

Ray greatly reduces the barrier to distributed programming, allowing the majority of Python developers and data scientists to focus on algorithms and application logic itself, rather than underlying distributed system details. If the tasks you need to handle exceed the processing capacity of a single machine, whether it's data processing, model training, or hyperparameter search, Ray is a very worthwhile and powerful tool to learn and use.

As a general-purpose distributed computing platform, the Ray framework has become core infrastructure in the large model ecosystem, achieving seamless integration with key tools like Hugging Face Transformers and DeepSpeed. Developers can use Ray to easily scale standard Hugging Face training code written for single GPUs to run on large-scale clusters with almost no modifications. At the same time, Ray can also act as a "commander" to schedule DeepSpeed, achieving distributed training of giant models far exceeding single-card VRAM capacity through its powerful memory optimization technology. In summary, Ray shields the underlying distributed complexity, allowing developers to conveniently combine and use these industry-leading tools on a unified platform, greatly simplifying the development and deployment of large-scale AI models.


## Ionet Ray Cluster Deployment


Ionet Ray service usage is basically similar to VM and container services - just rent machines. When programming, you need to use the Ray framework.

The basic deployment steps are as follows. Here we only write the key steps. Other steps can be set to default or selected according to your needs:

Step 1: Choose deployment cluster type, generally choose "general":

<img src='img/ray1.png' width='600' />

Step 2: Choose provider, also choose default "ionet":

<img src='img/ray2.png' width='600' />

Step 6: Choose GPU card type, 4090 is sufficient for learning:

<img src='img/ray3.png' width='600' />

Step Summary: Choose the number of GPUs. For learning, choose 1 or 2, time choose 1 hour:

<img src='img/ray5.png' width='600' />

In terms of pricing, Ray's GPU price is basically about 50% of VM or Docker, much cheaper.


## Ray Programming Basics

Ray's official documentation: https://docs.ray.io/en/latest/index.html

Ionet provides some quickly deployable environments for learning and using Ray.

The code examples from the official documentation can be used directly. Here, we'll introduce a few simple ones.。

### Hello World Example

Note: Please run the code below in a configured Ray environment.

In [6]:
# Control group: Ordinary Python code (executes serially, takes about 4 seconds):

import time

def slow_function(i):
    time.sleep(1)
    return i

results = []
for i in range(4):
    results.append(slow_function(i))

print(results)

[0, 1, 2, 3]


In [ ]:
# Code using Ray (executes in parallel, takes about 1 second):
import ray
import time

# Start Ray
ray.init()

# Add the @ray.remote decorator
@ray.remote
def slow_function(i):
    time.sleep(1)
    return i

# Asynchronously call using .remote()
futures = [slow_function.remote(i) for i in range(4)]

# Use ray.get() to retrieve all results
results = ray.get(futures)

print(results)

As can be seen, Ray programming is very simple. You only need to:

1. Add the `@ray.remote` decorator to your existing function.
2. Call the function with `.remote()` to execute it on a remote worker.
3. Use `ray.get()` to synchronously wait for the result.

Here is another example for testing GPU access:

### GPU Test Example

In [ ]:
# Control group: execution on the local server
import ray
import torch
import os

def check_gpu_on_ray_worker():
    """
    This function runs directly in the current process, not on a Ray worker.
    It checks for GPU availability on the local (login) server.
    """
    print("--- Log from the local server process ---")
    
    # Check if CUDA is actually available
    if not torch.cuda.is_available():
        error_message = "Error: CUDA is not available on this machine."
        print(error_message)
        return {"error": error_message}

    # Retrieve GPU information
    gpu_index = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(gpu_index)

    # Print logs from the local process for debugging
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: True")
    print(f"Process PID: {os.getpid()}")
    print(f"Running on GPU Index: {gpu_index}")
    print(f"GPU Name: {gpu_name}")
    print("------------------------------------")

    # Package the check results into a dictionary for return
    result = {
        "pytorch_version": torch.__version__,
        "cuda_available": True,
        "gpu_index": gpu_index,
        "gpu_name": gpu_name,
        "worker_pid": os.getpid()
    }
    return result


gpu_info = check_gpu_on_ray_worker()

print(gpu_info)

As shown in the output below, the printed result displays the information of the current login server, not our RTX 4090 worker server.

<img src='img/ray8.png' width='600' />


In [ ]:
import ray
import torch
import os

# Step 1: Wrap the GPU check code into a function
# Step 2: Use the @ray.remote decorator to turn it into a Ray Task
# Key: Use num_gpus=1 to tell Ray that this task must run on a node with a GPU
@ray.remote(num_gpus=1)
def check_gpu_on_ray_worker():
    """
    This function will be executed in an isolated Ray worker process.
    The Ray scheduler ensures this worker is assigned to a node with at least one GPU available.
    """
    print("--- Log from the Ray Worker process ---")
    
    # Check if CUDA is actually available
    if not torch.cuda.is_available():
        error_message = "Error: Ray assigned a GPU, but PyTorch cannot detect it."
        print(error_message)
        return {"error": error_message}

    # Retrieve GPU information
    gpu_index = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(gpu_index)
    
    # Print logs from the Worker side for debugging
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: True")
    print(f"Worker process PID: {os.getpid()}")
    print(f"Running on GPU Index: {gpu_index}")
    print(f"GPU Name: {gpu_name}")
    print("------------------------------------")

    # Package the check results into a dictionary for return
    result = {
        "pytorch_version": torch.__version__,
        "cuda_available": True,
        "gpu_index": gpu_index,
        "gpu_name": gpu_name,
        "worker_pid": os.getpid()
    }
    return result


# Initialize Ray (only once)
ray.init()

# Asynchronously execute the task using .remote()
# Ray will find a worker node that satisfies the num_gpus=1 requirement
check_future = check_gpu_on_ray_worker.remote()

# Step 5: Use ray.get() to retrieve the task result
# This will block until the remote function completes
gpu_info = ray.get(check_future)

# Print the dictionary returned from the Worker in a formatted way
print(gpu_info)

As shown below, the printed result displays the information of the Ray worker server, which is exactly our RTX 4090 worker server.

<img src='img/ray9.png' width='600' />

### Large Model Fine-tuning Example

Here, we use the official Ray example for fine-tuning a BERT-based large model for classification, and provide code for both local execution and Ray-distributed execution.

In [ ]:
# Example for local execution. Can be run on an Ionet VM, but DO NOT run on Ray's login server!!! It will block all tasks.
# To use the Ionet VM, install the following packages (can be done within Jupyter):
# pip install transformers[torch]
# pip install tf_keras evaluate


# Adapted from Hugging Face tutorial: https://huggingface.co/docs/transformers/training

import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# Datasets
dataset = load_dataset("yelp_review_full")
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

small_train_dataset = dataset["train"].select(range(10000)).map(tokenize_function, batched=True)
small_eval_dataset = dataset["test"].select(range(10000)).map(tokenize_function, batched=True)

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased", num_labels=5
)

# Metrics
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Hugging Face Trainer
training_args = TrainingArguments(
    output_dir="test_trainer"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

# Start Training
trainer.train()

Running on the Ionet VM as shown below:

<img src='img/ray10.png' width='600' />

In [ ]:
Then, the execution is carried out using Ray. It's important to note that since Ray is a distributed computing framework, the file storage system must be accessible by multiple nodes. It is generally recommended to use cloud storage for this purpose.

This includes services such as Amazon S3, Google Cloud Storage, and Alibaba Cloud OSS.

In this example, Amazon S3 is used, which requires prior registration and setup.

In [1]:
import os
import pyarrow.fs  # Ensure pyarrow is installed (pip install pyarrow)
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

import ray.train.huggingface.transformers
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer

import ray


"""
For gaming GPUs like RTX 4090, the following settings might be necessary.
For professional GPUs such as H100 or V100, these settings are not required.
"""
# Initialize Ray with runtime_env to upgrade pyarrow on workers
ray.init(
    runtime_env={
        "env_vars": {  # Add environment variables here for workers
            "NCCL_P2P_DISABLE": "1",
            "NCCL_IB_DISABLE": "1"
        }
    }
)

# [1] Encapsulate data preprocessing, training, and evaluation logic in a training function
# ==========================================================================================
def train_func():
    # Datasets
    dataset = load_dataset("yelp_review_full")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True)

    small_train_dataset = (
        dataset["train"].select(range(10000)).map(tokenize_function, batched=True)
    )
    small_eval_dataset = (
        dataset["test"].select(range(10000)).map(tokenize_function, batched=True)
    )

    # Model
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-cased", num_labels=5
    )

    # Evaluation Metrics
    metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)

    # Hugging Face Trainer
    training_args = TrainingArguments(
        output_dir="test_trainer",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=small_train_dataset,
        eval_dataset=small_eval_dataset,
        compute_metrics=compute_metrics,
    )

    # [2] Report Metrics and Checkpoints to Ray Train
    # =================================================
    callback = ray.train.huggingface.transformers.RayTrainReportCallback()
    trainer.add_callback(callback)

    # [3] Prepare Transformers Trainer
    # ==================================
    trainer = ray.train.huggingface.transformers.prepare_trainer(trainer)

    # Start Training
    trainer.train()


# Create custom S3 filesystem with credentials
fs = pyarrow.fs.S3FileSystem(
    access_key="******",  # Replace with your actual key
    secret_key="******",  # Replace with your actual secret
    region="us-west-1",  # Specify your bucket's region here
    # Optional: Add 'session_token="YOUR_SESSION_TOKEN"' if using temporary credentials
    # Optional: Add 'endpoint_override="https://s3.custom-endpoint.com"' for non-AWS S3-compatible storage like MinIO
)

# [4] Define a Ray TorchTrainer to launch `train_func` on all workers
# ====================================================================
ray_trainer = TorchTrainer(
    train_func,
    scaling_config=ScalingConfig(num_workers=2, use_gpu=True),
    run_config=RunConfig(
        # Use bucket/path without the 's3://' prefix when providing a custom filesystem
        storage_path="ionet2/ray1",
        storage_filesystem=fs,
        # Optional: Add 'name="your-experiment-name"' to organize runs
    ),
)
result: ray.train.Result = ray_trainer.fit()

# [5] Load the trained model
with result.checkpoint.as_directory() as checkpoint_dir:
    checkpoint_path = os.path.join(
        checkpoint_dir,
        ray.train.huggingface.transformers.RayTrainReportCallback.CHECKPOINT_NAME,
    )
    model = AutoModelFor

ModuleNotFoundError: No module named 'evaluate'

Please note that since some Ionet servers are multi-GPU machines, `num_workers=1` actually corresponds to the number of GPUs.

For example, even if there is only one worker node, if it has two GPUs, you should set `num_workers=2` to utilize both.

With the same code, the execution using 1 node/GPU looks like this:

<img src='img/ray11.png' width='600' />

And with two nodes:

<img src='img/ray12.png' width='600' />

As shown, training takes approximately 13 minutes with 1 GPU, but only about 7 minutes with 2 GPUs.

The code above provides a basic framework for training large models using Hugging Face Transformers. In most cases, you only need to change the model ID and dataset ID to train various other large models.

### S3-Compatible Cloud Storage

If you're using an S3-compatible cloud storage service, such as Alibaba Cloud OSS, the setup is essentially the same, though some specific configurations may be required.

Since the versions of Ray and PyArrow used in Ionet are relatively outdated, you may need to upgrade them within your program.

Here's an example using Alibaba Cloud OSS:

**Note:** First, upgrade PyArrow on the execution node:

```bash
!pip install pyarrow==16.0.0
```

Then, use a Ray script to upgrade PyArrow across all compute nodes.

In [3]:
import os
import pyarrow.fs  # Ensure pyarrow is upgraded locally: pip install --upgrade pyarrow or conda install -c conda-forge pyarrow>=16.0.0
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
import ray  # Import ray here

import ray.train.huggingface.transformers
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer


# Initialize Ray with runtime_env to upgrade pyarrow on workers
ray.init(
    runtime_env={
        "pip": ["pyarrow==16.0.0"],  # This will install/upgrade pyarrow on all worker nodes for this session
        "env_vars": {  # Add environment variables here for workers
            "NCCL_P2P_DISABLE": "1",
            "NCCL_IB_DISABLE": "1"
        }
    }
)

# [1] Encapsulate data preprocessing, training, and evaluation
# logic in a training function
# ============================================================
def train_func():
    # Datasets
    dataset = load_dataset("yelp_review_full")
    tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True)

    small_train_dataset = (
        dataset["train"].select(range(10000)).map(tokenize_function, batched=True)
    )
    small_eval_dataset = (
        dataset["test"].select(range(10000)).map(tokenize_function, batched=True)
    )

    # Model
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-cased", num_labels=5
    )

    # Evaluation Metrics
    metric = evaluate.load("accuracy")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)

    # Hugging Face Trainer
    training_args = TrainingArguments(
        output_dir="test_trainer",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=small_train_dataset,
        eval_dataset=small_eval_dataset,
        compute_metrics=compute_metrics,
    )

    # [2] Report Metrics and Checkpoints to Ray Train
    # ===============================================
    callback = ray.train.huggingface.transformers.RayTrainReportCallback()
    trainer.add_callback(callback)

    # [3] Prepare Transformers Trainer
    # ================================
    trainer = ray.train.huggingface.transformers.prepare_trainer(trainer)

    # Start Training
    trainer.train()


# Create custom S3-compatible filesystem for Alibaba OSS
fs = pyarrow.fs.S3FileSystem(
    access_key="*****",  # Your OSS Access Key ID
    secret_key="*****",  # Your OSS Secret Access Key
    endpoint_override="oss-cn-hongkong.aliyuncs.com",  # Global OSS endpoint (no scheme or bucket)
    scheme="https",  # Use HTTPS for security
    region="cn-hongkong",  # Your OSS region
    force_virtual_addressing=True,  # Force virtual-hosted style for OSS compatibility (requires PyArrow >=16.0)
)

# [4] Define a Ray TorchTrainer to launch `train_func` on all workers
# ===================================================================
ray_trainer = TorchTrainer(
    train_func,
    scaling_config=ScalingConfig(num_workers=2, use_gpu=True),
    run_config=RunConfig(
        # Include bucket in path
        storage_path="ionet/experiments/run1",  # Updated placeholder to a example path; adjust as needed
        storage_filesystem=fs,
        # Optional: Add 'name="your-experiment-name"' to organize runs
    ),
)
result: ray.train.Result = ray_trainer.fit()

# [5] Load the trained model
with result.checkpoint.as_directory() as checkpoint_dir:
    checkpoint_path = os.path.join(
        checkpoint_dir,
        ray.train.huggingface.transformers.RayTrainReportCallback.CHECKPOINT_NAME,
    )
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)

SyntaxError: unterminated string literal (detected at line 93) (451135922.py, line 93)